<a href="https://colab.research.google.com/github/batch15studios/DistillLab/blob/main/train_batchcode_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Headless Fine-Tuning: 'batchcode' (Qwen2.5-Coder-1.5B-Instruct)
### DistillLab Dataset Distillation Pipeline
- **Base Architecture**: `Qwen/Qwen2.5-Coder-1.5B-Instruct`
- **Dataset**: 441 Curated Multi-Trait Samples (Coding + Human Reasoning + Agentic Tool Calling)
- **Framework**: Unsloth 4-bit QLoRA
- **Quantized Export**: Direct GGUF (`q4_k_m`) for local execution in Ollama

In [1]:
# 1. Install Unsloth and optimized dependencies
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps trl peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 23.5 MB/s eta 0:00:00


In [2]:
# 2. Download latest multi-trait dataset directly from DistillLab repo
!curl -s -L -o cs_programming_dataset.jsonl "https://raw.githubusercontent.com/batch15studios/DistillLab/main/data/cs_programming_dataset.jsonl"

dataset_file = "cs_programming_dataset.jsonl"
with open(dataset_file, "r", encoding="utf-8") as f:
    content = f.read()

# Ensure zero preprocessor anomalies
cleaned = content.replace("#template", "template").replace("#requires", "requires").replace("#concept", "concept")
with open(dataset_file, "w", encoding="utf-8") as f:
    f.write(cleaned)

sample_count = len([line for line in cleaned.strip().split("\n") if line.strip()])
print(f"Verified dataset loaded: {sample_count} multi-trait samples ready for training.")

Verified dataset loaded: 441 multi-trait samples ready for training.


In [3]:
# 3. Load Qwen 2.5 Coder in 4-bit with LoRA adapters
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# Attach LoRA adapters to Attention + MLP layers
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.10.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [4]:
# 4. Apply ChatML template formatting
from datasets import load_dataset

def format_chatml(batch):
    texts = [tokenizer.apply_chat_template(convo, tokenize=False) for convo in batch["messages"]]
    return {"text": texts}

dataset = load_dataset("json", data_files=dataset_file, split="train")
dataset = dataset.map(format_chatml, batched=True)
print(f"Formatted {len(dataset)} samples with ChatML.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/441 [00:00<?, ? examples/s]

Formatted 441 samples with ChatML.


In [5]:
# 5. Train with SFTTrainer (Backpropagation)
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# Compute 2 full epochs with effective batch size 8 (1 per-device * 8 gradient accumulation)
training_steps = max(60, int((len(dataset) * 2) / 8))
print(f"Starting SFTTrainer for {training_steps} steps (~2 full epochs of {len(dataset)} samples)...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=max(5, int(training_steps * 0.05)),
        max_steps=training_steps,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="./outputs_batchcode",
        seed=3407,
    ),
)

trainer.train()
print("Backpropagation fine-tuning completed!")

Starting SFTTrainer for 110 steps (~2 full epochs of 441 samples)...


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/441 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/441 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 179 | Num Epochs = 5 | Total steps = 110
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,1.332174
10,1.237418
15,0.997593
20,0.982636
25,0.986341
30,0.862229
35,0.818880
40,0.912122
45,0.863420
50,0.900196


Unsloth: Restored added_tokens_decoder metadata in ./outputs_batchcode/checkpoint-110/tokenizer_config.json.


Backpropagation fine-tuning completed!


In [6]:
# 6. Direct Quantized GGUF Export for Ollama ('batchcode')
print("Merging LoRA adapters and exporting to 4-bit GGUF (q4_k_m)...")
model.save_pretrained_gguf("batchcode", tokenizer, quantization_method="q4_k_m")
print("GGUF quantization complete in ./batchcode/")

# Create Ollama Modelfile
modelfile = '''FROM ./unsloth.Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

SYSTEM """You are BatchCode, an elite Staff Systems Engineer, Analytical Theorist, and Agentic Coder.
Your core traits and execution standards:
1. Production Systems Code: Deliver robust, high-performance C, C++, Rust, Go, and Python solutions with zero placeholders, full boundary checks, and explicit Big-O Time & Space complexity.
2. Structured Human Reasoning: When diagnosing issues or analyzing architectures, provide systematic step-by-step root-cause analysis (RCA), trade-off evaluation, and invariant derivation. Talk naturally and collaboratively with the user.
3. Agentic Tool Calling: When a task requires inspecting files, running terminal commands, or searching code, output structured tool calls formatted as:
   <tool_call>
   {"name": "tool_name", "arguments": {"param": "value"}}
   </tool_call>
"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
'''

with open("batchcode/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print("Ollama Modelfile generated in batchcode/Modelfile")

Merging LoRA adapters and exporting to 4-bit GGUF (q4_k_m)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in batchcode/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:30<00:00, 30.89s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:02<00:00, 62.47s/it]


Unsloth: Merge process complete. Saved to `/content/batchcode`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11798-mix-f8a7950 (app-b11798-mix-f8a7950-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['batchcode_gguf/Qwen2.5-Coder-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed succ

In [7]:
# 7. Download quantized batchcode artifact
from google.colab import files
!zip -r batchcode_quantized.zip batchcode/
files.download('batchcode_quantized.zip')

  adding: batchcode/ (stored 0%)
  adding: batchcode/config.json (deflated 70%)
  adding: batchcode/tokenizer_config.json (deflated 83%)
  adding: batchcode/chat_template.jinja (deflated 71%)
  adding: batchcode/model.safetensors (deflated 21%)
  adding: batchcode/Modelfile (deflated 43%)
  adding: batchcode/generation_config.json (deflated 33%)
  adding: batchcode/tokenizer.json (deflated 81%)
  adding: batchcode/.cache/ (stored 0%)
  adding: batchcode/.cache/huggingface/ (stored 0%)
  adding: batchcode/.cache/huggingface/download/ (stored 0%)
  adding: batchcode/.cache/huggingface/download/tokenizer.model.lock (stored 0%)
  adding: batchcode/.cache/huggingface/download/model.safetensors.lock (stored 0%)
  adding: batchcode/.cache/huggingface/download/model.safetensors.metadata (deflated 31%)
  adding: batchcode/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
  adding: batchcode/.cache/huggingface/.gitignore (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>